# RUL observation-size sweep on the last clogging scenario

This notebook mirrors the methodology notebook's setup style, but instead of a
single assimilation run it **sweeps the number of field-data observations** fed to
the *last* clogging cycle's MCMC calibration, and reports **RUL distribution
statistics at each of the three prognostics stages**: `prior`, `bmu`, `bmu_enks`.

It can be used two ways:
1. **Locally / inline** - run the sweep in-process (small grids, quick checks).
2. **On a SLURM cluster** - one array task per observation size `k`, then aggregate.

In [ ]:
# Enable autoreload
%load_ext autoreload
%autoreload 2

import json
import os
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clogging_prognostics import (CloggingPipeline, PipelineConfig,
                                  run_observation_size,  build_base_surrogate, build_calibration_ensemble)

### MATPLOTLIB CONFIGURATION ###
plt.rcParams['figure.dpi'] = 150
# plt.rcParams['text.usetex'] = True   # enable if a LaTeX toolchain is available

In [ ]:
BASE_DIRECTORY = 'DATA/'
SG_TAG = 'SGB'

# Observation-size grid: number of field-data points kept in the LAST cycle.
OBS_SIZES = [0, 20, 40, 61]

# Prognostics-window anchors (hours), as in the methodology notebook.
SPLIT_SIM_HOURS = 176568
PROGNOSTICS_SPLIT_HOURS = 176808
N_RUL_SAMPLES = 1000

## Option 1 - Inline sweep (single process)

Good for small grids and debugging. Each `k` runs the full pipeline
(assimilation on all cycles, with the last cycle truncated to `k` observations),
then builds the prior / BMU / BMU+EnKS RUL ensembles and summarizes them.

In [ ]:
cfg = PipelineConfig(
    base_directory=BASE_DIRECTORY,
    input_dimension=7,
    n_doe_samples=1000,
    n_cycles=2,
    clogging_threshold=60.0,
)

cfg.metamodel.noise_floor = 1e-3   # the nugget — raise from 1e-4
cfg.metamodel.noise      = 1e-2    # keep init >= floor

pipe = CloggingPipeline(cfg).setup(sg_tag=SG_TAG)

In [ ]:
rows = []
for k in OBS_SIZES:
    print(f'=== k = {k} observations ===')
    stats = run_observation_size(
        pipe, k=k,
        split_sim_hours=SPLIT_SIM_HOURS,
        prognostics_split_hours=PROGNOSTICS_SPLIT_HOURS,
        n_rul_samples=N_RUL_SAMPLES,
    )
    for s in stats:
        rows.append(s.to_dict())
        print(f'  {s.stage:9s} mean={s.mean:9.1f}  std={s.std:9.1f}  '
              f'median={s.median:9.1f}  cross={s.count}/{s.n_total}')

df = pd.DataFrame(rows)
df.head(12)

In [ ]:
stages = ['prior', 'bmu', 'bmu_enks']
labels = {'prior': 'Prior', 'bmu': 'BMU', 'bmu_enks': 'BMU + EnKS'}
colors = {'prior': 'green', 'bmu': 'royalblue', 'bmu_enks': 'lightcoral'}

fig, ax = plt.subplots(figsize=(8, 5))
for stage in stages:
    d = df[df['stage'] == stage].sort_values('n_obs')
    ax.plot(d['n_obs'], d['mean'], '-o', color=colors[stage], label=labels[stage])
    ax.fill_between(d['n_obs'], d['q25'], d['q75'], color=colors[stage], alpha=0.2)

ax.set_xlabel('Number of observations in last cycle, $k$')
ax.set_ylabel('RUL')
ax.set_title('RUL mean and IQR vs. observation size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig.savefig(os.path.join(BASE_DIRECTORY, f'rul_vs_obs_size_{SG_TAG}.png'), dpi=300)

In [ ]:
# Convergence of RUL dispersion (coefficient of variation) with more data
fig, ax = plt.subplots(figsize=(8, 5))
for stage in stages:
    d = df[df['stage'] == stage].sort_values('n_obs')
    ax.plot(d['n_obs'], d['cv'], '-o', color=colors[stage], label=labels[stage])
ax.set_xlabel('Number of observations in last cycle, $k$')
ax.set_ylabel('RUL coefficient of variation')
ax.set_title('RUL relative dispersion vs. observation size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig.savefig(os.path.join(BASE_DIRECTORY, f'rul_cv_vs_obs_size_{SG_TAG}.png'), dpi=300)

## Option 2 - Cluster sweep (SLURM array)

Write the experiment config, then submit one array task per `k`. Aggregation
runs after the array finishes (via a dependency job or on the login node).

In [ ]:
# Write the experiment config consumed by run_experiment.py / submit_sweep.slurm
exp_config = {
    'sg_tag': SG_TAG,
    'outdir': 'results',
    'obs_sizes': OBS_SIZES,
    'pipeline': {
        'base_directory': BASE_DIRECTORY,
        'input_dimension': 7,
        'n_doe_samples': 1000,
        'train_size': 0.75,
        'n_cycles': 2,
        'clogging_threshold': 60.0,
    },
    'run_kwargs': {
        'split_sim_hours': SPLIT_SIM_HOURS,
        'prognostics_split_hours': PROGNOSTICS_SPLIT_HOURS,
        'n_rul_samples': N_RUL_SAMPLES,
    },
}
with open('exp_config.json', 'w') as f:
    json.dump(exp_config, f, indent=2)

print('set  #SBATCH --array=0-' + str(len(OBS_SIZES) - 1) + '  in submit_sweep.slurm')

Edit the `#SBATCH --array=` line in `submit_sweep.slurm` to `0-{len(OBS_SIZES)-1}`,
load your environment (module/conda lines), then submit. The cells below show the
typical launch pattern - uncomment to run from a login node with SLURM.

In [ ]:
# --- submit the array, then a dependent aggregation job ---
jid = subprocess.check_output(
    ['sbatch', '--parsable', 'submit_sweep.slurm']).decode().strip()
print('array job id:', jid)
subprocess.run(['sbatch', f'--dependency=afterok:{jid}', 'submit_aggregate.sh'])

In [ ]:
# --- or aggregate manually once the array has finished ---
# subprocess.run(['python', 'aggregate_results.py', '--outdir', 'results'])

## Load & inspect aggregated results

Works for either option: the inline sweep already produced `df`; the cluster path
produces `results/rul_summary.csv`. The cell below prefers the CSV if present.

In [ ]:
summary_csv = os.path.join('results', 'rul_summary.csv')
if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    print('loaded', summary_csv)
else:
    print('using inline df from Option 1')

df.sort_values(['stage', 'n_obs']).head(12)

## Plot: RUL statistics vs. number of observations

Mean RUL with an inter-quartile band, one curve per stage - the key figure showing
how the RUL estimate sharpens (and shifts) as more field data informs the last cycle.

In [ ]:
stages = ['prior', 'bmu', 'bmu_enks']
labels = {'prior': 'Prior', 'bmu': 'BMU', 'bmu_enks': 'BMU + EnKS'}
colors = {'prior': 'green', 'bmu': 'royalblue', 'bmu_enks': 'lightcoral'}

fig, ax = plt.subplots(figsize=(8, 5))
for stage in stages:
    d = df[df['stage'] == stage].sort_values('n_obs')
    ax.plot(d['n_obs'], d['mean'], '-o', color=colors[stage], label=labels[stage])
    ax.fill_between(d['n_obs'], d['q25'], d['q75'], color=colors[stage], alpha=0.2)

ax.set_xlabel('Number of observations in last cycle, $k$')
ax.set_ylabel('RUL')
ax.set_title('RUL mean and IQR vs. observation size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
# Convergence of RUL dispersion (coefficient of variation) with more data
fig, ax = plt.subplots(figsize=(8, 5))
for stage in stages:
    d = df[df['stage'] == stage].sort_values('n_obs')
    ax.plot(d['n_obs'], d['cv'], '-o', color=colors[stage], label=labels[stage])
ax.set_xlabel('Number of observations in last cycle, $k$')
ax.set_ylabel('RUL coefficient of variation')
ax.set_title('RUL relative dispersion vs. observation size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()